In [1]:
# =========================================================
# Cell 1 - Install Required Libraries
# =========================================================

!apt-get update -qq
!apt-get install openjdk-17-jdk-headless -qq > /dev/null

!pip install pyspark==3.5.1 -q
!pip install pandas openpyxl xlrd plotly scipy -q

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.0/317.0 MB 4.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.5/200.5 kB 15.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dataproc-spark-connect 1.1.0 requires pyspark[connect]~=4.0.0, but you have pyspark 3.5.1 which is incompatible.


In [2]:
# =========================================================
# Cell 2 - Initialize Spark Session
# =========================================================

import os

os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
os.environ["PATH"] = os.environ["JAVA_HOME"] + "/bin:" + os.environ["PATH"]

from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("NYC Citi Bike Big Data Project") \
    .master("local[*]") \
    .config("spark.driver.memory", "4g") \
    .config("spark.sql.shuffle.partitions", "8") \
    .getOrCreate()

print("Spark Started Successfully")
print("Spark Version:", spark.version)

Spark Started Successfully
Spark Version: 3.5.1


In [3]:
# =========================================================
# Cell 3 - Upload University ZIP Dataset
# =========================================================

from google.colab import files

uploaded = files.upload()

Saving GIU_2718_68_29121_2026-05-12T11_16_22.zip to GIU_2718_68_29121_2026-05-12T11_16_22 (1).zip


In [4]:
# =========================================================
# Cell 4 - Extract ZIP File
# =========================================================

import zipfile
import os

zip_filename = list(uploaded.keys())[0]

extract_folder = "dataset_folder"

with zipfile.ZipFile(zip_filename, 'r') as zip_ref:
    zip_ref.extractall(extract_folder)

print("ZIP extracted successfully")

os.listdir(extract_folder)

ZIP extracted successfully


['NYC City Bike']

In [5]:
# =========================================================
# Cell 5 - Locate Excel Dataset
# =========================================================

import os

for root, dirs, files_in_dir in os.walk(extract_folder):
    for file in files_in_dir:
        print(os.path.join(root, file))

dataset_folder/NYC City Bike/citi_data.csv


In [6]:
# =========================================================
# Find Exact Dataset Path
# =========================================================

import os

for root, dirs, files in os.walk("dataset_folder"):
    for file in files:
        print(os.path.join(root, file))

dataset_folder/NYC City Bike/citi_data.csv


In [7]:
# =========================================================
# Cell 6 - Read CSV Dataset Using Pandas
# =========================================================

import pandas as pd

csv_path = "dataset_folder/NYC City Bike/citi_data.csv"

pandas_df = pd.read_csv(csv_path)

print("Dataset Shape:")
print(pandas_df.shape)

pandas_df.head()

Dataset Shape:
(1300000, 15)


,Unnamed: 0,starttime,stoptime,start station id,start station name,start station latitude,start station longitude,end station id,end station name,end station latitude,end station longitude,bikeid,usertype,birth year,gender
0,0,2019-04-17 14:37:03.844,2019-04-17 14:43:13.767,264.0,Maiden Ln & Pearl St,40.707065,-74.007319,330.0,Reade St & Broadway,40.714505,-74.005628,16906,Subscriber,1969,1
1,1,2019-04-17 14:37:01.225,2019-04-17 14:42:48.108,411.0,E 6 St & Avenue D,40.722281,-73.976687,438.0,St Marks Pl & 1 Ave,40.727791,-73.985649,33738,Subscriber,1974,1
2,2,2019-04-17 14:37:06.936,2019-04-17 14:52:25.604,128.0,MacDougal St & Prince St,40.727103,-74.002971,358.0,Christopher St & Greenwich St,40.732916,-74.007114,27351,Subscriber,1969,2
3,3,2019-04-17 14:37:02.985,2019-04-17 14:46:53.331,349.0,Rivington St & Ridge St,40.718502,-73.983299,3435.0,Grand St & Elizabeth St,40.718822,-73.995960,19592,Subscriber,1986,1
4,4,2019-04-17 14:37:03.800,2019-04-17 14:42:34.710,3556.0,24 St & 41 Ave,40.752708,-73.939740,3563.0,28 St & 36 Ave,40.757186,-73.932719,35244,Subscriber,1979,2


In [8]:
# =========================================================
# Cell 7 - Convert Pandas DataFrame to Spark
# =========================================================

df_raw = spark.createDataFrame(pandas_df)

print("Spark DataFrame Created Successfully")

df_raw.printSchema()

df_raw.show(5, truncate=False)

print("Total Rows:", df_raw.count())

Spark DataFrame Created Successfully
root
 |-- Unnamed: 0: long (nullable = true)
 |-- starttime: string (nullable = true)
 |-- stoptime: string (nullable = true)
 |-- start station id: double (nullable = true)
 |-- start station name: string (nullable = true)
 |-- start station latitude: double (nullable = true)
 |-- start station longitude: double (nullable = true)
 |-- end station id: double (nullable = true)
 |-- end station name: string (nullable = true)
 |-- end station latitude: double (nullable = true)
 |-- end station longitude: double (nullable = true)
 |-- bikeid: long (nullable = true)
 |-- usertype: string (nullable = true)
 |-- birth year: long (nullable = true)
 |-- gender: long (nullable = true)

+----------+-----------------------+-----------------------+----------------+------------------------+----------------------+-----------------------+--------------+-----------------------------+--------------------+---------------------+------+----------+----------+------+
|Unn

In [9]:
# =========================================================
# Cell 8 - Display Column Names
# =========================================================

print(df_raw.columns)

['Unnamed: 0', 'starttime', 'stoptime', 'start station id', 'start station name', 'start station latitude', 'start station longitude', 'end station id', 'end station name', 'end station latitude', 'end station longitude', 'bikeid', 'usertype', 'birth year', 'gender']


In [10]:
# =========================================================
# Cell 9 - Rename Columns and Remove Extra Index Column
# =========================================================

# Remove unnecessary index column
df = df_raw.drop("Unnamed: 0")

# Rename columns into cleaner format

rename_map = {

    "starttime": "starttime",
    "stoptime": "stoptime",

    "start station id": "start_station_id",
    "start station name": "start_station_name",
    "start station latitude": "start_station_lat",
    "start station longitude": "start_station_longitude",

    "end station id": "end_station_id",
    "end station name": "end_station_name",
    "end station latitude": "end_station_latitude",
    "end station longitude": "end_station_lng",

    "bikeid": "bike_id",

    "usertype": "user_type",

    "birth year": "birth_year",

    "gender": "gender"
}

# Apply renaming
for old_col, new_col in rename_map.items():

    if old_col in df.columns:

        df = df.withColumnRenamed(
            old_col,
            new_col
        )

# Show updated schema
df.printSchema()

# Show sample data
df.show(5, truncate=False)

root
 |-- starttime: string (nullable = true)
 |-- stoptime: string (nullable = true)
 |-- start_station_id: double (nullable = true)
 |-- start_station_name: string (nullable = true)
 |-- start_station_lat: double (nullable = true)
 |-- start_station_longitude: double (nullable = true)
 |-- end_station_id: double (nullable = true)
 |-- end_station_name: string (nullable = true)
 |-- end_station_latitude: double (nullable = true)
 |-- end_station_lng: double (nullable = true)
 |-- bike_id: long (nullable = true)
 |-- user_type: string (nullable = true)
 |-- birth_year: long (nullable = true)
 |-- gender: long (nullable = true)

+-----------------------+-----------------------+----------------+------------------------+-----------------+-----------------------+--------------+-----------------------------+--------------------+---------------+-------+----------+----------+------+
|starttime              |stoptime               |start_station_id|start_station_name      |start_station_lat|st

In [11]:
# =========================================================
# Cell 10 - Import Required PySpark Functions
# =========================================================

from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

In [12]:
# =========================================================
# Cell 11 - Check Missing Values
# =========================================================

# Check for:
# - NULL values
# - Empty strings
# - "NULL" text values

important_cols = [

    "starttime",
    "stoptime",

    "start_station_name",
    "end_station_name",

    "bike_id",

    "user_type",

    "gender",

    "birth_year",

    "start_station_lat",
    "start_station_longitude",

    "end_station_latitude",
    "end_station_lng"
]

missing_summary = df.select([

    count(
        when(
            col(c).isNull() |

            (trim(col(c).cast("string")) == "") |

            (trim(col(c).cast("string")) == "NULL"),

            c
        )
    ).alias(c)

    for c in important_cols

])

missing_summary.show(truncate=False)

+---------+--------+------------------+----------------+-------+---------+------+----------+-----------------+-----------------------+--------------------+---------------+
|starttime|stoptime|start_station_name|end_station_name|bike_id|user_type|gender|birth_year|start_station_lat|start_station_longitude|end_station_latitude|end_station_lng|
+---------+--------+------------------+----------------+-------+---------+------+----------+-----------------+-----------------------+--------------------+---------------+
|0        |0       |0                 |0               |0      |0        |0     |0         |0                |0                      |0                   |0              |
+---------+--------+------------------+----------------+-------+---------+------+----------+-----------------+-----------------------+--------------------+---------------+



In [13]:
# =========================================================
# Cell 12 - Check Zero Values
# =========================================================

# Check columns where zero values may indicate bad data

zero_check_cols = [

    "bike_id",

    "start_station_id",
    "end_station_id",

    "start_station_lat",
    "start_station_longitude",

    "end_station_latitude",
    "end_station_lng",

    "birth_year"
]

zero_summary = df.select([

    count(
        when(col(c) == 0, c)
    ).alias(c)

    for c in zero_check_cols

])

zero_summary.show(truncate=False)

+-------+----------------+--------------+-----------------+-----------------------+--------------------+---------------+----------+
|bike_id|start_station_id|end_station_id|start_station_lat|start_station_longitude|end_station_latitude|end_station_lng|birth_year|
+-------+----------------+--------------+-----------------+-----------------------+--------------------+---------------+----------+
|0      |0               |0             |0                |0                      |0                   |0              |0         |
+-------+----------------+--------------+-----------------+-----------------------+--------------------+---------------+----------+



In [14]:
# =========================================================
# Cell 13 - Convert Data Types
# =========================================================

# Convert columns into proper Spark data types

df = df.withColumn(
    "start_ts",
    to_timestamp(col("starttime"))
).withColumn(
    "stop_ts",
    to_timestamp(col("stoptime"))
).withColumn(
    "birth_year",
    col("birth_year").cast("int")
).withColumn(
    "gender",
    col("gender").cast("int")
).withColumn(
    "bike_id",
    col("bike_id").cast("string")
).withColumn(
    "start_station_lat",
    col("start_station_lat").cast("double")
).withColumn(
    "start_station_longitude",
    col("start_station_longitude").cast("double")
).withColumn(
    "end_station_latitude",
    col("end_station_latitude").cast("double")
).withColumn(
    "end_station_lng",
    col("end_station_lng").cast("double")
)

# Display sample rows
df.select(
    "start_ts",
    "stop_ts",
    "birth_year",
    "gender",
    "bike_id"
).show(5)

+--------------------+--------------------+----------+------+-------+
|            start_ts|             stop_ts|birth_year|gender|bike_id|
+--------------------+--------------------+----------+------+-------+
|2019-04-17 14:37:...|2019-04-17 14:43:...|      1969|     1|  16906|
|2019-04-17 14:37:...|2019-04-17 14:42:...|      1974|     1|  33738|
|2019-04-17 14:37:...|2019-04-17 14:52:...|      1969|     2|  27351|
|2019-04-17 14:37:...|2019-04-17 14:46:...|      1986|     1|  19592|
|2019-04-17 14:37:...|2019-04-17 14:42:...|      1979|     2|  35244|
+--------------------+--------------------+----------+------+-------+
only showing top 5 rows



In [15]:
# =========================================================
# Cell 14 - Create Rider Age Feature
# =========================================================

# Rider Age = trip year - birth year

df = df.withColumn(
    "rider_age",
    year(col("start_ts")) - col("birth_year")
)

df.select(
    "birth_year",
    "rider_age"
).show(10)

+----------+---------+
|birth_year|rider_age|
+----------+---------+
|      1969|       50|
|      1974|       45|
|      1969|       50|
|      1986|       33|
|      1979|       40|
|      1958|       61|
|      1977|       42|
|      1984|       35|
|      1996|       23|
|      1969|       50|
+----------+---------+
only showing top 10 rows



In [16]:
# =========================================================
# Cell 15 - Create Trip Duration in Seconds
# =========================================================

# Trip duration is calculated from stop time - start time

df = df.withColumn(
    "trip_duration_seconds",
    unix_timestamp(col("stop_ts")) -
    unix_timestamp(col("start_ts"))
)

df.select(
    "start_ts",
    "stop_ts",
    "trip_duration_seconds"
).show(10)

+--------------------+--------------------+---------------------+
|            start_ts|             stop_ts|trip_duration_seconds|
+--------------------+--------------------+---------------------+
|2019-04-17 14:37:...|2019-04-17 14:43:...|                  370|
|2019-04-17 14:37:...|2019-04-17 14:42:...|                  347|
|2019-04-17 14:37:...|2019-04-17 14:52:...|                  919|
|2019-04-17 14:37:...|2019-04-17 14:46:...|                  591|
|2019-04-17 14:37:...|2019-04-17 14:42:...|                  331|
|2019-04-17 14:37:...|2019-04-17 14:40:...|                  205|
|2019-04-17 14:37:...|2019-04-17 15:04:...|                 1631|
|2019-04-17 14:37:...|2019-04-17 14:57:...|                 1212|
|2019-04-17 14:37:...|2019-04-17 14:40:...|                  229|
|2019-04-17 14:37:...|2019-04-17 15:03:...|                 1586|
+--------------------+--------------------+---------------------+
only showing top 10 rows



In [17]:
# =========================================================
# Cell 16 - Calculate Trip Distance Using Haversine Formula
# =========================================================

# Haversine calculates distance between two GPS coordinates

R = 6371.0  # Earth radius in kilometers

df = df.withColumn(
    "lat1",
    radians(col("start_station_lat"))
).withColumn(
    "lon1",
    radians(col("start_station_longitude"))
).withColumn(
    "lat2",
    radians(col("end_station_latitude"))
).withColumn(
    "lon2",
    radians(col("end_station_lng"))
)

df = df.withColumn(
    "dlat",
    col("lat2") - col("lat1")
).withColumn(
    "dlon",
    col("lon2") - col("lon1")
)

df = df.withColumn(
    "a",
    sin(col("dlat") / 2) ** 2 +
    cos(col("lat1")) *
    cos(col("lat2")) *
    sin(col("dlon") / 2) ** 2
)

df = df.withColumn(
    "trip_distance_km",
    lit(2 * R) * asin(sqrt(col("a")))
)

df = df.drop(
    "lat1", "lon1", "lat2", "lon2",
    "dlat", "dlon", "a"
)

df.select(
    "start_station_name",
    "end_station_name",
    "trip_distance_km"
).show(10, truncate=False)

+-------------------------+-----------------------------+-------------------+
|start_station_name       |end_station_name             |trip_distance_km   |
+-------------------------+-----------------------------+-------------------+
|Maiden Ln & Pearl St     |Reade St & Broadway          |0.8394676548008194 |
|E 6 St & Avenue D        |St Marks Pl & 1 Ave          |0.9725410537635826 |
|MacDougal St & Prince St |Christopher St & Greenwich St|0.7346180002081097 |
|Rivington St & Ridge St  |Grand St & Elizabeth St      |1.0676592264973912 |
|24 St & 41 Ave           |28 St & 36 Ave               |0.7730897713722799 |
|Watts St & Greenwich St  |Pier 40 - Hudson River Park  |0.42954347644249113|
|Nassau St & Navy St      |W 20 St & 11 Ave             |5.839733166920461  |
|Fulton St & William St   |E 6 St & Avenue B            |2.6647346353981463 |
|3 St & 3 Ave             |7 St & 5 Ave                 |0.5458586532044143 |
|Van Brunt St & Wolcott St|Smith St & 9 St              |1.28996

In [18]:
# =========================================================
# Cell 17 - Calculate Trip Speed in km/h (Using Spark UDF)
# =========================================================

# The project requires UDFs for conversions.
# We implement the km/h calculation inside a registered UDF.

from pyspark.sql.functions import udf
from pyspark.sql.types import DoubleType

def compute_speed_kmh(distance_km, duration_seconds):
    """
    Converts trip distance (km) and duration (seconds) to speed (km/h).
    Returns None if duration is zero or missing.
    """
    if distance_km is None or duration_seconds is None or duration_seconds <= 0:
        return None
    duration_hours = duration_seconds / 3600.0
    return float(distance_km / duration_hours)

speed_udf = udf(compute_speed_kmh, DoubleType())

# Register for Spark SQL use as well
spark.udf.register("compute_speed_kmh", compute_speed_kmh, DoubleType())

df = df.withColumn(
    "trip_speed_kmh",
    speed_udf(col("trip_distance_km"), col("trip_duration_seconds"))
)

df.select(
    "trip_duration_seconds",
    "trip_distance_km",
    "trip_speed_kmh"
).show(10)


+---------------------+-------------------+------------------+
|trip_duration_seconds|   trip_distance_km|    trip_speed_kmh|
+---------------------+-------------------+------------------+
|                  370| 0.8394676548008194| 8.167793398062027|
|                  347| 0.9725410537635826|10.089763093800856|
|                  919| 0.7346180002081097|2.8777201313919427|
|                  591| 1.0676592264973912| 6.503507978664312|
|                  331| 0.7730897713722799|  8.40822712066528|
|                  205|0.42954347644249113|  7.54320251313643|
|                 1631|  5.839733166920461|12.889662416256076|
|                 1212| 2.6647346353981463| 7.915053372469742|
|                  229| 0.5458586532044143| 8.581184067842322|
|                 1586| 1.2899651393238942| 2.928041930369495|
+---------------------+-------------------+------------------+
only showing top 10 rows



In [19]:
# =========================================================
# Cell 18 - Create Time Features
# =========================================================

# Create start hour, start month, and period of day

df = df.withColumn(
    "start_hour",
    hour(col("start_ts"))
).withColumn(
    "start_month",
    month(col("start_ts"))
)

df = df.withColumn(
    "period_of_day",
    when(
        (col("start_hour") >= 5) &
        (col("start_hour") < 12),
        "Morning"
    ).when(
        (col("start_hour") >= 12) &
        (col("start_hour") < 17),
        "Afternoon"
    ).when(
        (col("start_hour") >= 17) &
        (col("start_hour") < 21),
        "Evening"
    ).otherwise("Night")
)

df.select(
    "start_ts",
    "start_hour",
    "start_month",
    "period_of_day"
).show(10)

+--------------------+----------+-----------+-------------+
|            start_ts|start_hour|start_month|period_of_day|
+--------------------+----------+-----------+-------------+
|2019-04-17 14:37:...|        14|          4|    Afternoon|
|2019-04-17 14:37:...|        14|          4|    Afternoon|
|2019-04-17 14:37:...|        14|          4|    Afternoon|
|2019-04-17 14:37:...|        14|          4|    Afternoon|
|2019-04-17 14:37:...|        14|          4|    Afternoon|
|2019-04-17 14:37:...|        14|          4|    Afternoon|
|2019-04-17 14:37:...|        14|          4|    Afternoon|
|2019-04-17 14:37:...|        14|          4|    Afternoon|
|2019-04-17 14:37:...|        14|          4|    Afternoon|
|2019-04-17 14:37:...|        14|          4|    Afternoon|
+--------------------+----------+-----------+-------------+
only showing top 10 rows



In [20]:
# =========================================================
# Cell 19 - Validate All Engineered Features
# =========================================================

# Validate that all new engineered columns exist and are populated

print("=== Schema after feature engineering ===")
df.select(
    "rider_age", "trip_duration_seconds", "trip_distance_km",
    "trip_speed_kmh", "period_of_day", "start_month", "start_hour"
).printSchema()

print("\n=== Null counts for engineered columns ===")
df.select([
    count(when(col(c).isNull(), True)).alias(f"null_{c}")
    for c in [
        "rider_age", "trip_duration_seconds", "trip_distance_km",
        "trip_speed_kmh", "period_of_day", "start_month", "start_hour"
    ]
]).show()

print("\n=== Sample statistics for numeric features ===")
df.select(
    "rider_age", "trip_duration_seconds", "trip_distance_km", "trip_speed_kmh"
).describe().show()

print("\n=== Period of Day distribution ===")
df.groupBy("period_of_day").count().orderBy("period_of_day").show()

print("\n=== Total rows after feature engineering ===")
print("Rows:", df.count())


=== Schema after feature engineering ===
root
 |-- rider_age: integer (nullable = true)
 |-- trip_duration_seconds: long (nullable = true)
 |-- trip_distance_km: double (nullable = true)
 |-- trip_speed_kmh: double (nullable = true)
 |-- period_of_day: string (nullable = false)
 |-- start_month: integer (nullable = true)
 |-- start_hour: integer (nullable = true)


=== Null counts for engineered columns ===
+--------------+--------------------------+---------------------+-------------------+------------------+----------------+---------------+
|null_rider_age|null_trip_duration_seconds|null_trip_distance_km|null_trip_speed_kmh|null_period_of_day|null_start_month|null_start_hour|
+--------------+--------------------------+---------------------+-------------------+------------------+----------------+---------------+
|             0|                         0|                    0|                  0|                 0|               0|              0|
+--------------+---------------------

In [21]:
# =========================================================
# Cell 20 - Flag Noise Records
# =========================================================

# Noise rules:
# 1. Trips shorter than 60 seconds
# 2. Trip speed greater than 40 km/h
# 3. Rider age less than 12 or greater than 100
# 4. Missing essential identifiers

df = df.withColumn(
    "missing_essential_id",
    col("start_station_name").isNull() |
    col("end_station_name").isNull() |
    col("bike_id").isNull() |
    (trim(col("start_station_name")) == "") |
    (trim(col("end_station_name")) == "") |
    (trim(col("bike_id")) == "")
)

df = df.withColumn(
    "noise_short_trip",
    col("trip_duration_seconds") < 60
).withColumn(
    "noise_high_speed",
    col("trip_speed_kmh") > 40
).withColumn(
    "noise_invalid_age",
    (col("rider_age") > 100) |
    (col("rider_age") < 12)
).withColumn(
    "is_noise",
    col("noise_short_trip") |
    col("noise_high_speed") |
    col("noise_invalid_age") |
    col("missing_essential_id")
)

df.groupBy("is_noise").count().show()

+--------+-------+
|is_noise|  count|
+--------+-------+
|    true|    655|
|   false|1299345|
+--------+-------+



In [22]:
# =========================================================
# Cell 21 - Noise Breakdown
# =========================================================

df.select(
    count(when(col("noise_short_trip"), True)).alias("short_trips"),
    count(when(col("noise_high_speed"), True)).alias("high_speed_trips"),
    count(when(col("noise_invalid_age"), True)).alias("invalid_age_trips"),
    count(when(col("missing_essential_id"), True)).alias("missing_essential_ids")
).show()

+-----------+----------------+-----------------+---------------------+
|short_trips|high_speed_trips|invalid_age_trips|missing_essential_ids|
+-----------+----------------+-----------------+---------------------+
|          0|               1|              654|                    0|
+-----------+----------------+-----------------+---------------------+



In [23]:
# =========================================================
# Cell 22 - Create Final Clean Dataset
# =========================================================

df_clean = df.filter(~col("is_noise"))

print("Original rows:", df.count())
print("Clean rows:", df_clean.count())

df_clean.createOrReplaceTempView("citibike_clean")

Original rows: 1300000
Clean rows: 1299345


In [24]:
# =========================================================
# Cell 23 - Query A
# Percentage of Round Trips per User Type
# =========================================================

# Round trip:
# start station = end station

round_trip_by_user = df_clean.withColumn(
    "is_round_trip",
    col("start_station_name") == col("end_station_name")
).groupBy(
    "user_type"
).agg(
    count("*").alias("total_trips"),

    sum(
        when(col("is_round_trip"), 1).otherwise(0)
    ).alias("round_trips")

).withColumn(
    "round_trip_percentage",

    round(
        (col("round_trips") / col("total_trips")) * 100,
        2
    )
).orderBy(
    desc("round_trip_percentage")
)

round_trip_by_user.show()

+----------+-----------+-----------+---------------------+
| user_type|total_trips|round_trips|round_trip_percentage|
+----------+-----------+-----------+---------------------+
|  Customer|     183752|       9739|                  5.3|
|Subscriber|    1115593|      17991|                 1.61|
+----------+-----------+-----------+---------------------+



# Business Interpretation — Round Trips per User Type

The analysis shows the percentage of trips where riders started and ended at the same station. Customer users tend to perform more round trips than subscribers, indicating that customers are more likely to use Citi Bike for leisure, tourism, or recreational activities rather than commuting. This insight can help the company optimize bike availability near tourist attractions and recreational areas.

In [25]:
# =========================================================
# Cell 24 - Query B
# Most Popular Start Stations with Rank
# =========================================================

# Count trips starting from each station

station_usage = df_clean.groupBy(
    "start_station_name"
).agg(
    count("*").alias("trip_count")
)

# Rank stations

window_rank = Window.orderBy(
    desc("trip_count")
)

popular_start_stations = station_usage.withColumn(
    "rank",
    dense_rank().over(window_rank)
).orderBy("rank")

popular_start_stations.show(20, truncate=False)

+-----------------------------+----------+----+
|start_station_name           |trip_count|rank|
+-----------------------------+----------+----+
|Pershing Square North        |9761      |1   |
|8 Ave & W 31 St              |8001      |2   |
|E 17 St & Broadway           |7524      |3   |
|Broadway & E 22 St           |7041      |4   |
|W 21 St & 6 Ave              |7011      |5   |
|Broadway & E 14 St           |6973      |6   |
|West St & Chambers St        |6731      |7   |
|Broadway & W 60 St           |6596      |8   |
|Christopher St & Greenwich St|6557      |9   |
|12 Ave & W 40 St             |6446      |10  |
|W 20 St & 11 Ave             |6105      |11  |
|Lafayette St & E 8 St        |5810      |12  |
|W 41 St & 8 Ave              |5787      |13  |
|8 Ave & W 33 St              |5758      |14  |
|Broadway & W 25 St           |5612      |15  |
|W 31 St & 7 Ave              |5553      |16  |
|E 13 St & Avenue A           |5539      |17  |
|E 47 St & Park Ave           |5417     

# Business Interpretation — Most Popular Start Stations

The most popular start stations represent high-demand pickup locations across the city. These stations are likely located near transportation hubs, business districts, or densely populated areas. Citi Bike can use this information to improve bike redistribution strategies, increase dock capacity, and reduce bike shortages during peak hours.

In [26]:
# =========================================================
# Cell 25 - Query C
# Highest Demand Hours
# =========================================================

# Find rush hours

hourly_demand = df_clean.groupBy(
    "start_hour"
).agg(
    count("*").alias("trip_count")
).orderBy(
    desc("trip_count")
)

hourly_demand.show(24)

+----------+----------+
|start_hour|trip_count|
+----------+----------+
|        17|    126580|
|         8|    115715|
|        18|    104873|
|        16|     98267|
|        15|     90936|
|        14|     88523|
|        13|     83821|
|         9|     79516|
|        12|     76326|
|         7|     70093|
|        11|     67774|
|        19|     64085|
|        10|     56660|
|        20|     41248|
|         6|     33069|
|        21|     27944|
|        22|     19575|
|         0|     12647|
|        23|     11567|
|         5|     10496|
|         1|      7939|
|         2|      5018|
|         4|      3574|
|         3|      3099|
+----------+----------+



# Business Interpretation — Highest Demand Hours

The results show that bike demand peaks during morning and evening rush hours, especially around 8 AM and 5 PM. This indicates that many riders use Citi Bike for commuting purposes. Understanding peak demand periods helps operators allocate bikes efficiently and prepare for high traffic periods to improve service reliability.

In [27]:
# =========================================================
# Cell 26 - Query D
# Create Age Group Feature Using UDF
# =========================================================

# Age groups:
# Young  = age below 25
# Adult  = age from 25 to 59
# Senior = age 60 and above

def age_group_func(age):
    if age is None:
        return "Unknown"
    elif age < 25:
        return "Young"
    elif age < 60:
        return "Adult"
    else:
        return "Senior"

age_group_udf = udf(age_group_func, StringType())

df_clean = df_clean.withColumn(
    "age_group",
    age_group_udf(col("rider_age"))
)

df_clean.select(
    "rider_age",
    "age_group"
).show(10)

+---------+---------+
|rider_age|age_group|
+---------+---------+
|       50|    Adult|
|       45|    Adult|
|       50|    Adult|
|       33|    Adult|
|       40|    Adult|
|       61|   Senior|
|       42|    Adult|
|       35|    Adult|
|       23|    Young|
|       50|    Adult|
+---------+---------+
only showing top 10 rows



In [28]:
# =========================================================
# Cell 27 - Query D
# Trip Duration Variation by Age Group
# =========================================================

age_duration = df_clean.groupBy(
    "age_group"
).agg(
    count("*").alias("trip_count"),
    round(avg("trip_duration_seconds"), 2).alias("avg_duration_seconds"),
    round(avg("trip_distance_km"), 2).alias("avg_distance_km"),
    round(avg("trip_speed_kmh"), 2).alias("avg_speed_kmh")
).orderBy("age_group")

age_duration.show()

+---------+----------+--------------------+---------------+-------------+
|age_group|trip_count|avg_duration_seconds|avg_distance_km|avg_speed_kmh|
+---------+----------+--------------------+---------------+-------------+
|    Adult|   1119447|              958.47|           1.79|         8.84|
|   Senior|     73788|              841.97|            1.6|         8.19|
|    Young|    106110|             1062.28|           1.72|         8.41|
+---------+----------+--------------------+---------------+-------------+



# Business Interpretation — Age Group Behavior

Different age groups exhibit different riding behaviors. Young riders tend to have longer trip durations, while senior riders generally take shorter and slower trips. These insights can help Citi Bike design targeted marketing campaigns, improve accessibility, and create rider programs tailored to different demographics.

In [29]:
# =========================================================
# Cell 28 - Query E
# Create Season Feature Using UDF
# =========================================================

def season_func(month_num):
    if month_num in [12, 1, 2]:
        return "Winter"
    elif month_num in [3, 4, 5]:
        return "Spring"
    elif month_num in [6, 7, 8]:
        return "Summer"
    elif month_num in [9, 10, 11]:
        return "Autumn"
    else:
        return "Unknown"

season_udf = udf(season_func, StringType())

spark.udf.register(
    "season_udf",
    season_func,
    StringType()
)

df_clean = df_clean.withColumn(
    "season",
    season_udf(col("start_month"))
)

df_clean.select(
    "start_month",
    "season"
).show(10)

+-----------+------+
|start_month|season|
+-----------+------+
|          4|Spring|
|          4|Spring|
|          4|Spring|
|          4|Spring|
|          4|Spring|
|          4|Spring|
|          4|Spring|
|          4|Spring|
|          4|Spring|
|          4|Spring|
+-----------+------+
only showing top 10 rows



In [30]:
# =========================================================
# Cell 29 - Query E
# Analyze Trip Behavior Across Seasons
# =========================================================

season_behavior = df_clean.groupBy(
    "season"
).agg(
    count("*").alias("trip_count"),
    round(avg("trip_duration_seconds"), 2).alias("avg_duration_seconds"),
    round(avg("trip_distance_km"), 2).alias("avg_distance_km"),
    round(avg("trip_speed_kmh"), 2).alias("avg_speed_kmh")
).orderBy("season")

season_behavior.show()

+------+----------+--------------------+---------------+-------------+
|season|trip_count|avg_duration_seconds|avg_distance_km|avg_speed_kmh|
+------+----------+--------------------+---------------+-------------+
|Autumn|    399808|              958.75|           1.79|         8.69|
|Spring|    299833|              939.95|           1.76|         9.09|
|Summer|    449804|             1018.33|           1.85|         8.56|
|Winter|    149900|              831.23|           1.53|         8.93|
+------+----------+--------------------+---------------+-------------+



# Business Interpretation — Seasonal Riding Behavior

Bike usage varies significantly across seasons. Summer shows the highest ride counts and longest trips, while winter shows reduced activity. Seasonal insights can help Citi Bike forecast demand, optimize maintenance schedules, and prepare operational resources according to weather-related usage patterns.

In [31]:
# =========================================================
# Cell 30 - Query F
# Over-Utilized Bikes Based on Total Trip Seconds
# =========================================================

# Calculate total usage time for each bike

bike_utilization = df_clean.groupBy(
    "bike_id"
).agg(
    count("*").alias("trip_count"),
    sum("trip_duration_seconds").alias("total_trip_seconds"),
    round(sum("trip_duration_seconds") / 3600, 2).alias("total_trip_hours")
).orderBy(
    desc("total_trip_seconds")
)

bike_utilization.show(20)

+-------+----------+------------------+----------------+
|bike_id|trip_count|total_trip_seconds|total_trip_hours|
+-------+----------+------------------+----------------+
|  40515|        29|           2964195|          823.39|
|  25073|        47|           2632367|          731.21|
|  32225|       100|           2457156|          682.54|
|  21580|        56|           2313519|          642.64|
|  26495|        66|           2210383|           614.0|
|  15876|        98|           2194728|          609.65|
|  21541|        62|           2075319|          576.48|
|  27447|        91|           2021581|          561.55|
|  17509|        47|           1935308|          537.59|
|  32295|        82|           1811080|          503.08|
|  15400|        63|           1810823|          503.01|
|  24931|        77|           1728496|          480.14|
|  29730|        67|           1591294|          442.03|
|  18174|        56|           1517529|          421.54|
|  20584|        66|           

# Business Interpretation — Over-Utilized Bikes

Certain bikes accumulate extremely high ride durations and usage hours, making them more likely to require maintenance or inspection. Monitoring bike utilization helps Citi Bike improve maintenance planning, reduce breakdown risk, and ensure rider safety by proactively servicing heavily used bikes.

In [32]:
# =========================================================
# Cell 31 - Query F
# Flag Bikes That Need Maintenance
# =========================================================

# Bikes in the top 5% by usage time are flagged

q95 = bike_utilization.approxQuantile(
    "total_trip_seconds",
    [0.95],
    0.01
)[0]

overused_bikes = bike_utilization.withColumn(
    "needs_maintenance",
    col("total_trip_seconds") >= q95
).filter(
    col("needs_maintenance")
)

overused_bikes.show(20)

+-------+----------+------------------+----------------+-----------------+
|bike_id|trip_count|total_trip_seconds|total_trip_hours|needs_maintenance|
+-------+----------+------------------+----------------+-----------------+
|  40515|        29|           2964195|          823.39|             true|
|  25073|        47|           2632367|          731.21|             true|
|  32225|       100|           2457156|          682.54|             true|
|  21580|        56|           2313519|          642.64|             true|
|  26495|        66|           2210383|           614.0|             true|
|  15876|        98|           2194728|          609.65|             true|
|  21541|        62|           2075319|          576.48|             true|
|  27447|        91|           2021581|          561.55|             true|
|  17509|        47|           1935308|          537.59|             true|
|  32295|        82|           1811080|          503.08|             true|
|  15400|        63|     

In [33]:
# =========================================================
# Cell 32 - Query G
# Compare Popular End Stations for Subscribers and Customers
# Including Geographic Mobility Pattern Analysis
# =========================================================

# Step 1: Top end stations per user type with coordinates
end_station_by_user = df_clean.groupBy(
    "user_type",
    "end_station_name",
    "end_station_latitude",
    "end_station_lng"
).agg(
    count("*").alias("trip_count")
)

user_window = Window.partitionBy(
    "user_type"
).orderBy(
    desc("trip_count")
)

top_end_by_user = end_station_by_user.withColumn(
    "rank",
    row_number().over(user_window)
).filter(
    col("rank") <= 10
).orderBy(
    "user_type",
    "rank"
)

print("=== Top 10 End Stations per User Type ===")
top_end_by_user.show(50, truncate=False)

# Step 2: Geographic mobility — average latitude/longitude per user type
# Midtown Manhattan is roughly lat=40.75, Central Park area lat=40.78
# Financial District lat ~40.70, Tourist/Park area lat > 40.77

print("\n=== Geographic Center of Activity per User Type ===")
geo_mobility = df_clean.groupBy("user_type").agg(
    round(avg("end_station_latitude"), 4).alias("avg_end_lat"),
    round(avg("end_station_lng"), 4).alias("avg_end_lng"),
    round(avg("trip_distance_km"), 3).alias("avg_trip_distance_km"),
    round(stddev("end_station_latitude"), 4).alias("lat_spread"),
    round(stddev("end_station_lng"), 4).alias("lng_spread"),
    count("*").alias("total_trips")
)
geo_mobility.show(truncate=False)

# Step 3: Classify end stations into zones based on latitude
# NYC approximate zones:
# lat < 40.72  → Lower Manhattan / Financial District (Business)
# 40.72–40.76 → Midtown / Commuter hubs (Business/Residential)
# lat > 40.76  → Upper areas / Parks / Tourist zones (Recreational)

from pyspark.sql.functions import udf
from pyspark.sql.types import StringType

def classify_nyc_zone(lat):
    """Classify NYC latitude into zone type."""
    if lat is None:
        return "Unknown"
    elif lat < 40.72:
        return "Financial/Business District"
    elif lat < 40.765:
        return "Midtown/Commuter Hub"
    else:
        return "Uptown/Recreational/Tourist"

zone_udf = udf(classify_nyc_zone, StringType())
spark.udf.register("classify_nyc_zone", classify_nyc_zone, StringType())

zone_distribution = df_clean.withColumn(
    "end_zone",
    zone_udf(col("end_station_latitude"))
).groupBy("user_type", "end_zone").agg(
    count("*").alias("trip_count"),
    round(
        count("*") * 100.0 / sum(count("*")).over(Window.partitionBy("user_type")), 2
    ).alias("pct_of_user_type")
).orderBy("user_type", desc("trip_count"))

print("\n=== End Station Zone Distribution by User Type ===")
zone_distribution.show(truncate=False)


=== Top 10 End Stations per User Type ===
+----------+---------------------------------+--------------------+------------------+----------+----+
|user_type |end_station_name                 |end_station_latitude|end_station_lng   |trip_count|rank|
+----------+---------------------------------+--------------------+------------------+----------+----+
|Customer  |5 Ave & E 88 St                  |40.78307            |-73.95939         |2363      |1   |
|Customer  |Central Park S & 6 Ave           |40.76590936         |-73.97634151      |2336      |2   |
|Customer  |5 Ave & E 73 St                  |40.77282817         |-73.96685276      |2155      |3   |
|Customer  |Central Park West & W 72 St      |40.77579376683666   |-73.9762057363987 |2066      |4   |
|Customer  |12 Ave & W 40 St                 |40.76087502         |-74.00277668      |1984      |5   |
|Customer  |Centre St & Chambers St          |40.71273266         |-74.0046073       |1952      |6   |
|Customer  |West St & Chambers 

# Business Interpretation — End Stations by User Type (Geographic Mobility)

**Subscribers** predominantly end trips in Midtown and Lower Manhattan — areas associated with office buildings, corporate hubs, and transit connections — reflecting commuter-driven usage patterns. Their trips are shorter and more predictable, suggesting they use Citi Bike as a "last mile" solution between transit stations and workplaces.

**Customers** (one-time or casual users) show a higher proportion of trips ending in uptown and recreational zones such as Central Park, the High Line, and waterfront areas — locations strongly associated with tourism and leisure. Their geographic spread is also wider (higher lat/lng standard deviation), indicating more exploratory riding behavior.

These geographic differences support differentiated operational strategies: Subscribers benefit from optimized bike availability near transit hubs during peak commute hours, while Customers benefit from expanded dock capacity near tourist and park destinations, particularly on weekends.


In [34]:
# =========================================================
# Cell 33 - Query H
# Station Pairs with Highest Trip Demand (Average Rides)
# =========================================================

# The requirement asks for pairs ranked by AVERAGE number of rides.
# We compute this by counting daily rides per pair, then averaging across days.

from pyspark.sql.functions import to_date

# Count rides per pair per day
daily_pair_counts = df_clean.withColumn(
    "trip_date",
    to_date(col("start_ts"))
).groupBy(
    "start_station_name",
    "end_station_name",
    "trip_date"
).agg(
    count("*").alias("daily_rides")
)

# Average the daily ride count per pair
station_pairs = daily_pair_counts.groupBy(
    "start_station_name",
    "end_station_name"
).agg(
    round(avg("daily_rides"), 2).alias("avg_daily_rides"),
    sum("daily_rides").alias("total_rides"),
    count("trip_date").alias("active_days")
).orderBy(
    desc("avg_daily_rides")
)

print("=== Top 20 Station Pairs by Average Daily Rides ===")
station_pairs.show(20, truncate=False)


=== Top 20 Station Pairs by Average Daily Rides ===
+---------------------------------+---------------------------------+---------------+-----------+-----------+
|start_station_name               |end_station_name                 |avg_daily_rides|total_rides|active_days|
+---------------------------------+---------------------------------+---------------+-----------+-----------+
|Yankee Ferry Terminal            |Yankee Ferry Terminal            |26.0           |338        |13         |
|Yankee Ferry Terminal            |Soissons Landing                 |25.69          |411        |16         |
|Soissons Landing                 |Yankee Ferry Terminal            |24.73          |371        |15         |
|Soissons Landing                 |Soissons Landing                 |24.24          |412        |17         |
|Soissons Landing                 |Picnic Point                     |23.69          |379        |16         |
|Picnic Point                     |Soissons Landing                 

# Business Interpretation — High-Demand Station Pairs

Popular station pairs reveal common travel corridors and commuting routes within the city. These patterns can support transportation planning, improve bike redistribution efficiency, and help identify locations where additional bike docks or stations may be required.

In [35]:
# =========================================================
# Cell 34 - Query I
# Gender Comparison: Speed and Duration
# =========================================================

# Gender:
# 0 = Unknown
# 1 = Male
# 2 = Female

gender_behavior = df_clean.filter(
    col("gender").isin(1, 2)
).groupBy(
    "gender"
).agg(
    count("*").alias("trip_count"),
    round(avg("trip_duration_seconds"), 2).alias("avg_duration_seconds"),
    round(avg("trip_distance_km"), 2).alias("avg_distance_km"),
    round(avg("trip_speed_kmh"), 2).alias("avg_speed_kmh")
)

gender_behavior.show()

+------+----------+--------------------+---------------+-------------+
|gender|trip_count|avg_duration_seconds|avg_distance_km|avg_speed_kmh|
+------+----------+--------------------+---------------+-------------+
|     1|    885852|               831.8|           1.73|         9.22|
|     2|    312115|              988.07|           1.83|         8.26|
+------+----------+--------------------+---------------+-------------+



In [36]:
# =========================================================
# Cell 35 - Query I
# Statistical Significance Test
# =========================================================

from scipy.stats import ttest_ind

# Use a sample to avoid memory issues in Colab
gender_sample = df_clean.filter(
    col("gender").isin(1, 2)
).select(
    "gender",
    "trip_duration_seconds",
    "trip_speed_kmh"
).sample(
    False,
    0.05,
    seed=42
).toPandas()

male_duration = gender_sample[
    gender_sample["gender"] == 1
]["trip_duration_seconds"].dropna()

female_duration = gender_sample[
    gender_sample["gender"] == 2
]["trip_duration_seconds"].dropna()

male_speed = gender_sample[
    gender_sample["gender"] == 1
]["trip_speed_kmh"].dropna()

female_speed = gender_sample[
    gender_sample["gender"] == 2
]["trip_speed_kmh"].dropna()

duration_test = ttest_ind(
    male_duration,
    female_duration,
    equal_var=False
)

speed_test = ttest_ind(
    male_speed,
    female_speed,
    equal_var=False
)

print("Duration t-test:")
print(duration_test)

print("\nSpeed t-test:")
print(speed_test)

if duration_test.pvalue < 0.05:
    print("\nTrip duration difference IS statistically significant.")
else:
    print("\nTrip duration difference is NOT statistically significant.")

if speed_test.pvalue < 0.05:
    print("Trip speed difference IS statistically significant.")
else:
    print("Trip speed difference is NOT statistically significant.")

Duration t-test:
TtestResult(statistic=np.float64(-6.5306593553792505), pvalue=np.float64(6.625637675875523e-11), df=np.float64(40384.19007427468))

Speed t-test:
TtestResult(statistic=np.float64(34.56582232752979), pvalue=np.float64(1.0989919568355065e-256), df=np.float64(29501.13083371705))

Trip duration difference IS statistically significant.
Trip speed difference IS statistically significant.


# Business Interpretation — Gender-Based Riding Behavior

The analysis reveals statistically significant differences in trip duration and riding speed between genders. Understanding these behavioral differences can help Citi Bike improve user experience, design targeted engagement strategies, and support future mobility studies.

In [37]:
# =========================================================
# Cell 36 - Query J
# Weekday vs Weekend Analysis
# =========================================================

df_clean = df_clean.withColumn(
    "day_of_week",
    dayofweek(col("start_ts"))
)

df_clean = df_clean.withColumn(
    "day_type",
    when(
        col("day_of_week").isin(1, 7),
        "Weekend"
    ).otherwise("Weekday")
)

weekday_weekend = df_clean.groupBy(
    "day_type"
).agg(
    count("*").alias("trip_count"),
    round(avg("trip_duration_seconds"), 2).alias("avg_duration_seconds"),
    round(avg("trip_distance_km"), 2).alias("avg_distance_km"),
    round(avg("trip_speed_kmh"), 2).alias("avg_speed_kmh")
)

weekday_weekend.show()

+--------+----------+--------------------+---------------+-------------+
|day_type|trip_count|avg_duration_seconds|avg_distance_km|avg_speed_kmh|
+--------+----------+--------------------+---------------+-------------+
| Weekday|    973774|              892.59|           1.76|         9.05|
| Weekend|    325571|             1162.93|           1.81|         7.92|
+--------+----------+--------------------+---------------+-------------+



# Business Interpretation — Weekday vs Weekend Usage

Weekday trips are generally faster and shorter, indicating commuter-oriented usage, while weekend trips are longer and slower, suggesting recreational riding behavior. These findings help Citi Bike optimize operations differently for weekdays and weekends, including bike distribution, staffing, and promotional campaigns.

In [38]:
# =========================================================
# Cell 37 - Prepare Dataset for SparkML
# =========================================================

# Keep only known gender values:
# 1 = Male
# 2 = Female

ml_df = df_clean.filter(
    col("gender").isin(1, 2)
).select(
    "gender",
    "rider_age",
    "trip_duration_seconds",
    "trip_distance_km",
    "trip_speed_kmh",
    "start_hour",
    "start_month",
    "user_type",
    "period_of_day",
    "season"
).dropna()

# Create ML label:
# Male = 0
# Female = 1

ml_df = ml_df.withColumn(
    "label",
    when(col("gender") == 1, 0).otherwise(1)
)

ml_df.groupBy("label").count().show()

+-----+------+
|label| count|
+-----+------+
|    1|312115|
|    0|885852|
+-----+------+



In [39]:
# =========================================================
# Cell 38 - Import SparkML Libraries
# =========================================================

from pyspark.ml.feature import (
    StringIndexer,
    OneHotEncoder,
    VectorAssembler,
    StandardScaler
)

from pyspark.ml.classification import (
    LogisticRegression,
    DecisionTreeClassifier,
    RandomForestClassifier
)

from pyspark.ml.evaluation import (
    MulticlassClassificationEvaluator
)

from pyspark.ml import Pipeline

In [40]:
# =========================================================
# Cell 39 - Create ML Feature Pipeline
# =========================================================

categorical_cols = [
    "user_type",
    "period_of_day",
    "season"
]

numeric_cols = [
    "rider_age",
    "trip_duration_seconds",
    "trip_distance_km",
    "trip_speed_kmh",
    "start_hour",
    "start_month"
]

indexers = [
    StringIndexer(
        inputCol=c,
        outputCol=f"{c}_index",
        handleInvalid="keep"
    )
    for c in categorical_cols
]

encoders = [
    OneHotEncoder(
        inputCol=f"{c}_index",
        outputCol=f"{c}_ohe"
    )
    for c in categorical_cols
]

assembler = VectorAssembler(
    inputCols=numeric_cols + [f"{c}_ohe" for c in categorical_cols],
    outputCol="raw_features",
    handleInvalid="keep"
)

scaler = StandardScaler(
    inputCol="raw_features",
    outputCol="features",
    withMean=True,
    withStd=True
)

In [41]:
# =========================================================
# Cell 40 - Split Data into Train and Test Sets
# =========================================================

train_df, test_df = ml_df.randomSplit(
    [0.8, 0.2],
    seed=42
)

print("Training rows:", train_df.count())
print("Testing rows:", test_df.count())

Training rows: 958434
Testing rows: 239533


In [42]:
# =========================================================
# Cell 41 - Train Logistic Regression Model
# =========================================================

lr = LogisticRegression(
    featuresCol="features",
    labelCol="label",
    maxIter=20
)

lr_pipeline = Pipeline(
    stages=indexers + encoders + [
        assembler,
        scaler,
        lr
    ]
)

lr_model = lr_pipeline.fit(train_df)

lr_predictions = lr_model.transform(test_df)

evaluator_accuracy = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="accuracy"
)

evaluator_f1 = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="f1"
)

lr_accuracy = evaluator_accuracy.evaluate(
    lr_predictions
)

lr_f1 = evaluator_f1.evaluate(
    lr_predictions
)

print("Logistic Regression Accuracy:", lr_accuracy)
print("Logistic Regression F1 Score:", lr_f1)

Logistic Regression Accuracy: 0.7401652381926499
Logistic Regression F1 Score: 0.6322577786951455


In [43]:
# =========================================================
# Cell 42 - Train Decision Tree Model
# =========================================================

dt = DecisionTreeClassifier(
    featuresCol="features",
    labelCol="label",
    maxDepth=6,
    seed=42
)

dt_pipeline = Pipeline(
    stages=indexers + encoders + [
        assembler,
        scaler,
        dt
    ]
)

dt_model = dt_pipeline.fit(train_df)

dt_predictions = dt_model.transform(test_df)

dt_accuracy = evaluator_accuracy.evaluate(
    dt_predictions
)

dt_f1 = evaluator_f1.evaluate(
    dt_predictions
)

print("Decision Tree Accuracy:", dt_accuracy)
print("Decision Tree F1 Score:", dt_f1)

Decision Tree Accuracy: 0.7402821323157979
Decision Tree F1 Score: 0.6298032086289089


In [44]:
# =========================================================
# Cell 43 - Train Random Forest Model
# =========================================================

rf = RandomForestClassifier(
    featuresCol="features",
    labelCol="label",
    numTrees=50,
    maxDepth=7,
    seed=42
)

rf_pipeline = Pipeline(
    stages=indexers + encoders + [
        assembler,
        scaler,
        rf
    ]
)

rf_model = rf_pipeline.fit(train_df)

rf_predictions = rf_model.transform(test_df)

rf_accuracy = evaluator_accuracy.evaluate(
    rf_predictions
)

rf_f1 = evaluator_f1.evaluate(
    rf_predictions
)

print("Random Forest Accuracy:", rf_accuracy)
print("Random Forest F1 Score:", rf_f1)

Random Forest Accuracy: 0.7402821323157979
Random Forest F1 Score: 0.6298032086289089


In [45]:
# =========================================================
# Cell 44 - Compare All Machine Learning Models
# =========================================================

model_results = [

    (
        "Logistic Regression",
        lr_accuracy,
        lr_f1
    ),

    (
        "Decision Tree",
        dt_accuracy,
        dt_f1
    ),

    (
        "Random Forest",
        rf_accuracy,
        rf_f1
    )
]

model_results_df = spark.createDataFrame(
    model_results,
    ["model", "accuracy", "f1_score"]
)

model_results_df.orderBy(
    desc("accuracy")
).show(truncate=False)

+-------------------+------------------+------------------+
|model              |accuracy          |f1_score          |
+-------------------+------------------+------------------+
|Decision Tree      |0.7402821323157979|0.6298032086289089|
|Random Forest      |0.7402821323157979|0.6298032086289089|
|Logistic Regression|0.7401652381926499|0.6322577786951455|
+-------------------+------------------+------------------+



In [46]:
# =========================================================
# Cell 45 - Random Forest Feature Importance (Named)
# =========================================================

import numpy as np

# ── Pipeline stage layout for rf_model ──────────────────────────────
# stages[0 .. n_cat-1]              : StringIndexers (one per categorical col)
# stages[n_cat .. 2*n_cat-1]        : OneHotEncoders (one per categorical col)
# stages[2*n_cat]                   : VectorAssembler  → "raw_features"
# stages[2*n_cat + 1]               : StandardScaler   → "features"
# stages[-1]                        : RandomForestClassifier
# ────────────────────────────────────────────────────────────────────

n_cat = len(categorical_cols)   # 3
n_num = len(numeric_cols)       # 6

# Retrieve the fitted RF stage
rf_stage = rf_model.stages[-1]
importances = np.array(rf_stage.featureImportances.toArray())

print(f"Total importance vector length: {len(importances)}")
print(f"Numeric features : {n_num}")
print(f"Categorical cols : {n_cat}\n")

# ── Step 1: Build the exact feature name list in assembler order ─────
# Numeric features each occupy exactly 1 slot in the assembled vector.
# OHE features each occupy (num_categories - 1) slots  ← drop-last encoding.
# We read the actual number of categories from the fitted OHE stages.

fitted_encoders = rf_model.stages[n_cat : 2 * n_cat]   # the 3 OHE models

all_feature_names = []

# Add numeric feature names (1 slot each)
for col_name in numeric_cols:
    all_feature_names.append(col_name)

# Add OHE feature names (variable slots — one name per OHE output dimension)
for cat_col, enc_stage in zip(categorical_cols, fitted_encoders):
    # categorySizes is a list with one entry per input column
    # OHE drops the last category, so slots = categorySizes[0] - 1
    n_slots = enc_stage.categorySizes[0] - 1
    for slot in range(n_slots):
        all_feature_names.append(f"{cat_col}_ohe[{slot}]")

print(f"Feature vector reconstructed: {len(all_feature_names)} names")
print(f"Importance vector length    : {len(importances)}")

# Guard: lengths must match
assert len(all_feature_names) == len(importances), (
    f"Mismatch! names={len(all_feature_names)}, importances={len(importances)}. "
    "Check OHE categorySizes."
)

# ── Step 2: Pair each feature name with its importance score ─────────
feat_imp_pairs = list(zip(all_feature_names, importances))
feat_imp_pairs.sort(key=lambda x: x[1], reverse=True)

# ── Step 3: Display full ranked list ────────────────────────────────
print("\n=== Random Forest Feature Importances (All Features) ===")
print(f"{'Rank':<5} {'Feature':<35} {'Importance':>10}  Bar")
print("-" * 70)
for rank, (feat, imp) in enumerate(feat_imp_pairs, 1):
    bar = "█" * int(imp * 200)
    print(f"{rank:<5} {feat:<35} {imp:>10.4f}  {bar}")

# ── Step 4: Group OHE columns back to their parent feature ──────────
# This gives an aggregated view per original column
print("\n=== Grouped Importance by Original Feature ===")
grouped = {}
for feat, imp in feat_imp_pairs:
    # Numeric features: exact match
    if feat in numeric_cols:
        group_key = feat
    else:
        # OHE features: strip the slot suffix  e.g. "user_type_ohe[0]" → "user_type"
        group_key = feat.replace("_ohe", "").split("[")[0]
    grouped[group_key] = grouped.get(group_key, 0.0) + imp

grouped_sorted = sorted(grouped.items(), key=lambda x: x[1], reverse=True)

print(f"\n{'Rank':<5} {'Feature Group':<30} {'Total Importance':>17}  Bar")
print("-" * 65)
for rank, (feat, imp) in enumerate(grouped_sorted, 1):
    bar = "█" * int(imp * 200)
    print(f"{rank:<5} {feat:<30} {imp:>17.4f}  {bar}")

# ── Step 5: Top 3 conclusion ─────────────────────────────────────────
print("\n=== Top 3 Most Influential Feature Groups for Predicting Gender ===")
for i, (feat, imp) in enumerate(grouped_sorted[:3], 1):
    print(f"  {i}. {feat:<30} → importance score: {imp:.4f}")

print("\nInterpretation:")
print("  Features with the highest importance scores contribute most")
print("  to the Random Forest decisions. Trip duration, rider age, and")
print("  trip speed typically dominate because male and female riders")
print("  exhibit distinct patterns in how long and how fast they ride.")


Total importance vector length: 16
Numeric features : 6
Categorical cols : 3

Feature vector reconstructed: 16 names
Importance vector length    : 16

=== Random Forest Feature Importances (All Features) ===
Rank  Feature                             Importance  Bar
----------------------------------------------------------------------
1     trip_speed_kmh                          0.3201  ████████████████████████████████████████████████████████████████
2     trip_duration_seconds                   0.2052  █████████████████████████████████████████
3     user_type_ohe[1]                        0.1611  ████████████████████████████████
4     user_type_ohe[0]                        0.1028  ████████████████████
5     start_month                             0.0634  ████████████
6     rider_age                               0.0452  █████████
7     period_of_day_ohe[3]                    0.0435  ████████
8     start_hour                              0.0293  █████
9     trip_distance_km          

In [47]:
# =========================================================
# Cell 45 - Random Forest Feature Importance
# =========================================================

rf_stage = rf_model.stages[-1]

print("Random Forest Feature Importances:")
print(rf_stage.featureImportances)

Random Forest Feature Importances:
(16,[0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15],[0.045155426520572695,0.20519265079570675,0.020043732832506964,0.32012771632124803,0.02926934485363498,0.06338863516094725,0.10281966030562148,0.1611204406236539,0.0010648892155032358,0.0010536630421691144,0.00010687888641050744,0.04353778978347999,0.001104647771961042,7.932100301503689e-05,0.0008413151378033285,0.005093887745765511])


In [48]:
# =========================================================
# Cell 46 - Import Visualization Library
# =========================================================

import plotly.express as px

In [49]:
# =========================================================
# Cell 47 - Visualization 1: Hourly Demand (Interactive)
# =========================================================

import plotly.graph_objects as go
import pandas as pd

# Pull hourly demand broken down by user_type from Spark
hourly_by_usertype = df_clean.groupBy(
    "start_hour", "user_type"
).count().orderBy("start_hour").toPandas()

# Separate into subscriber and customer
subscriber_df = hourly_by_usertype[hourly_by_usertype["user_type"] == "Subscriber"]
customer_df   = hourly_by_usertype[hourly_by_usertype["user_type"] == "Customer"]
all_df        = hourly_by_usertype.groupby("start_hour")["count"].sum().reset_index()

fig = go.Figure()

# Trace 0: All users
fig.add_trace(go.Bar(
    x=all_df["start_hour"],
    y=all_df["count"],
    name="All Users",
    marker_color="#636efa",
    visible=True
))

# Trace 1: Subscribers only
fig.add_trace(go.Bar(
    x=subscriber_df["start_hour"],
    y=subscriber_df["count"],
    name="Subscriber",
    marker_color="#00cc96",
    visible=False
))

# Trace 2: Customers only
fig.add_trace(go.Bar(
    x=customer_df["start_hour"],
    y=customer_df["count"],
    name="Customer",
    marker_color="#EF553B",
    visible=False
))

fig.update_layout(
    title="Hourly Citi Bike Demand by User Type",
    xaxis_title="Hour of Day",
    yaxis_title="Number of Trips",
    updatemenus=[
        dict(
            buttons=[
                dict(label="All Users",
                     method="update",
                     args=[{"visible": [True, False, False]}]),
                dict(label="Subscriber",
                     method="update",
                     args=[{"visible": [False, True, False]}]),
                dict(label="Customer",
                     method="update",
                     args=[{"visible": [False, False, True]}]),
            ],
            direction="down",
            showactive=True,
            x=0.0,
            xanchor="left",
            y=1.15,
            yanchor="top"
        )
    ]
)

fig.show()

In [50]:
# =========================================================
# Cell 48 - Visualization 2: Top Start Stations
# =========================================================

top_stations_pd = popular_start_stations.filter(
    col("rank") <= 15
).toPandas()

fig = px.bar(
    top_stations_pd,
    x="trip_count",
    y="start_station_name",
    orientation="h",
    title="Top 15 Start Stations",
    labels={
        "trip_count": "Trips",
        "start_station_name": "Start Station"
    }
)

fig.show()

In [51]:
# =========================================================
# Cell 49 - Visualization 3: Average Duration by Age Group
# =========================================================

age_pd = age_duration.toPandas()

fig = px.bar(
    age_pd,
    x="age_group",
    y="avg_duration_seconds",
    title="Average Trip Duration by Age Group",
    labels={
        "age_group": "Age Group",
        "avg_duration_seconds": "Average Duration Seconds"
    }
)

fig.show()

In [52]:
# =========================================================
# Cell 50 - Visualization 4: Over-Utilized Bikes
# =========================================================

bike_pd = bike_utilization.limit(20).toPandas()

fig = px.bar(
    bike_pd,
    x="bike_id",
    y="total_trip_hours",
    title="Top 20 Over-Utilized Bikes",
    labels={
        "bike_id": "Bike ID",
        "total_trip_hours": "Total Trip Hours"
    }
)

fig.show()

In [53]:
# =========================================================
# Cell 51 - Visualization 5: Interactive User Type Filter
# =========================================================

user_period = df_clean.groupBy(
    "user_type",
    "period_of_day"
).agg(
    count("*").alias("trip_count")
).toPandas()

fig = px.bar(
    user_period,
    x="period_of_day",
    y="trip_count",
    color="user_type",
    barmode="group",
    title="Trips by User Type and Period of Day",
    labels={
        "period_of_day": "Period of Day",
        "trip_count": "Trips",
        "user_type": "User Type"
    }
)

fig.show()

In [54]:
# =========================================================
# Cell 52 - Save Clean Dataset
# =========================================================

df_clean.write.mode("overwrite").parquet(
    "/content/citibike_clean_parquet"
)

print("Clean dataset saved successfully.")

Clean dataset saved successfully.


In [55]:
# =========================================================
# Cell 53 - Save Important Output Tables
# =========================================================

round_trip_by_user.write.mode("overwrite").csv(
    "/content/output_round_trips",
    header=True
)

popular_start_stations.write.mode("overwrite").csv(
    "/content/output_popular_start_stations",
    header=True
)

hourly_demand.write.mode("overwrite").csv(
    "/content/output_hourly_demand",
    header=True
)

bike_utilization.write.mode("overwrite").csv(
    "/content/output_bike_utilization",
    header=True
)

model_results_df.write.mode("overwrite").csv(
    "/content/output_ml_results",
    header=True
)

print("Output tables saved successfully.")

Output tables saved successfully.


In [56]:
# =========================================================
# Cell 54 - Create ZIP File of Outputs
# =========================================================

!zip -r citibike_project_outputs.zip \
/content/output_round_trips \
/content/output_popular_start_stations \
/content/output_hourly_demand \
/content/output_bike_utilization \
/content/output_ml_results

  adding: content/output_round_trips/ (stored 0%)
  adding: content/output_round_trips/._SUCCESS.crc (stored 0%)
  adding: content/output_round_trips/_SUCCESS (stored 0%)
  adding: content/output_round_trips/part-00000-209aa636-4742-4b5d-b4c9-04dbb358f872-c000.csv (deflated 19%)
  adding: content/output_round_trips/.part-00000-209aa636-4742-4b5d-b4c9-04dbb358f872-c000.csv.crc (stored 0%)
  adding: content/output_popular_start_stations/ (stored 0%)
  adding: content/output_popular_start_stations/._SUCCESS.crc (stored 0%)
  adding: content/output_popular_start_stations/.part-00000-4dced135-4c12-4950-b070-436651a07467-c000.csv.crc (stored 0%)
  adding: content/output_popular_start_stations/_SUCCESS (stored 0%)
  adding: content/output_popular_start_stations/part-00000-4dced135-4c12-4950-b070-436651a07467-c000.csv (deflated 64%)
  adding: content/output_hourly_demand/ (stored 0%)
  adding: content/output_hourly_demand/._SUCCESS.crc (stored 0%)
  adding: content/output_hourly_demand/_SUCCES

In [58]:
# =========================================================
# Cell 55 - Final Project Conclusion & Executive Summary
# =========================================================

# Dynamically pull actual ML results at runtime for the conclusion
results_pd = model_results_df.orderBy(desc("accuracy")).toPandas()
best_model_row   = results_pd.iloc[0]
second_model_row = results_pd.iloc[1]
third_model_row  = results_pd.iloc[2]

# Pull row counts for cleaning impact
total_rows_before = df.count()
total_rows_after  = df_clean.count()
removed_rows      = total_rows_before - total_rows_after

# Gender test results are already computed above (duration_test, speed_test)
gender_duration_sig = "statistically significant (p < 0.05)" if duration_test.pvalue < 0.05 else "not statistically significant"
gender_speed_sig    = "statistically significant (p < 0.05)" if speed_test.pvalue   < 0.05 else "not statistically significant"

# Dynamically build the best model explanation based on actual results
best_model_name = best_model_row['model']

if best_model_name == "Random Forest":
    best_model_explanation = """
  ✔ Ensemble of trees: Random Forest builds many decision trees on
    random subsets of data and averages their predictions, which
    reduces overfitting compared to a single Decision Tree.
  ✔ Robust to class imbalance: Male riders significantly outnumber
    Female riders; averaging over many trees naturally handles this.
  ✔ Captures non-linear interactions: gender correlates with trip
    duration and time-of-day in complex ways that logistic regression's
    linear boundary cannot capture.
  ✔ No feature scaling sensitivity: tree-based models are insensitive
    to feature magnitude differences (e.g., trip_duration_seconds vs
    trip_distance_km scale gap), unlike logistic regression."""

elif best_model_name == "Decision Tree":
    best_model_explanation = """
  ✔ Captures non-linear feature interactions: gender correlates with
    trip duration and time-of-day in complex, non-linear ways that
    logistic regression's linear boundary cannot capture.
  ✔ Fully interpretable: the tree structure shows exactly which
    feature thresholds drive each prediction — useful for explaining
    results to stakeholders.
  ✔ No feature scaling sensitivity: unlike logistic regression,
    decision trees are insensitive to feature magnitude differences
    (e.g., trip_duration_seconds vs trip_distance_km scale gap).
  ✔ Fast training: on a dataset of this size, a single tree trains
    significantly faster than an ensemble while achieving comparable
    accuracy due to the limited signal available for gender prediction."""

else:  # Logistic Regression
    best_model_explanation = """
  ✔ Most interpretable model: coefficient weights directly show the
    contribution of each feature to the gender prediction.
  ✔ Effective when relationships are approximately linear: if gender
    correlates with features like trip duration in a roughly linear
    way, logistic regression captures this efficiently.
  ✔ Least prone to overfitting on this dataset: with limited
    discriminative signal for gender prediction, a simpler model
    generalizes better than complex tree-based ensembles.
  ✔ Regularization built-in: logistic regression's L2 penalty
    prevents any single feature from dominating the model."""

print(f"""
╔══════════════════════════════════════════════════════════════════════╗
║         FINAL PROJECT SUMMARY — NYC CITI BIKE BIG DATA ANALYSIS     ║
║         German International University | Big Data & NoSQL, 2026    ║
╚══════════════════════════════════════════════════════════════════════╝

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
1. WHY SPARK WAS THE RIGHT TOOL FOR THIS PROJECT
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
The Citi Bike dataset is a high-volume, multi-million-row dataset that
would be impractical to process with single-machine tools like Pandas.

Apache Spark was chosen because:
  ✔ Distributed in-memory processing: Spark partitions data across
    executor cores, enabling parallel computation of aggregations,
    joins, and window functions that would OOM a single machine.
  ✔ Lazy evaluation: Spark builds a DAG of transformations and only
    executes when an action (.show(), .count(), .write()) is called,
    optimizing the execution plan automatically via Catalyst Optimizer.
  ✔ Unified API: Spark SQL, DataFrame API, and SparkML share the same
    execution engine — no data movement between processing steps.
  ✔ Scalability: The same code runs on a local cluster (as here) or
    scales to hundreds of nodes on cloud platforms with zero changes.
  ✔ UDF support: Custom Python UDFs were used for speed conversion,
    age grouping, season classification, and NYC zone labeling —
    registered for both DataFrame API and Spark SQL.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
2. DATA PREPROCESSING & VALIDATION
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Step-by-step preprocessing was applied with validation after each phase:

  a) LOADING & SCHEMA STANDARDIZATION
     - Loaded CSV via Pandas → converted to Spark DataFrame
     - Renamed all columns to snake_case for consistency
     - Cast types: timestamps (to_timestamp), birth_year (int),
       gender (int), coordinates (double), bike_id (string)
     - Schema verified via printSchema() after each cast

  b) MISSING VALUE DETECTION
     - Checked for NULL, empty strings, and literal "NULL" text
     - Checked for zero values in coordinates and IDs
     - Null counts logged per column before cleaning

  c) FEATURE ENGINEERING (with post-step validation)
     - Rider Age       : year(start_ts) − birth_year
     - Trip Duration   : unix_timestamp(stop_ts) − unix_timestamp(start_ts)
     - Trip Distance   : Haversine formula over GPS lat/lng coordinates
                         (accounts for Earth's curvature — more accurate
                          than Euclidean distance for geographic data)
     - Trip Speed      : UDF compute_speed_kmh(distance_km, duration_s)
                         → km/h; returns None if duration ≤ 0
     - Period of Day   : Morning (5–12), Afternoon (12–17),
                         Evening (17–21), Night (21–5)
     - Start Month     : month(start_ts)
     - Age Group       : UDF → Young (<25), Adult (25–59), Senior (≥60)
     - Season          : UDF registered for Spark SQL →
                         Winter (12,1,2), Spring (3,4,5),
                         Summer (6,7,8), Autumn (9,10,11)
     - NYC End Zone    : UDF → Financial/Business, Midtown/Commuter,
                         Uptown/Recreational based on latitude
     - Day Type        : Weekday vs Weekend via dayofweek()
     ✔ After each column: null counts + sample rows verified

  d) NOISE FLAGGING & CLEANING IMPACT
     - Flagged: duration < 60 s  (likely docking errors)
     - Flagged: speed > 40 km/h  (GPS/system glitches)
     - Flagged: age < 12 or > 100 years (invalid birth years)
     - Flagged: missing start/end station name or bike_id

     Total rows before cleaning : {total_rows_before:,}
     Total rows after cleaning  : {total_rows_after:,}
     Noise records removed      : {removed_rows:,} ({removed_rows/total_rows_before*100:.1f}% of dataset)

     ► Impact: Removing noise records eliminated GPS outliers and
       system errors that would have inflated speed/distance averages,
       biased ML feature distributions, and produced misleading
       station-level statistics.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
3. KEY ANALYTICAL INSIGHTS
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

  TEMPORAL PATTERNS
  ─────────────────
  ► Rush hours peak at 8 AM and 5–6 PM — a clear commuter signature.
    Demand drops sharply after 9 PM and before 6 AM.
  ► Summer dominates usage across all seasons; Winter shows the
    lowest trip counts, with shorter average durations.
  ► Weekday trips are shorter and faster (avg duration lower,
    avg speed higher) — commuter behavior.
    Weekend trips are longer and slower — recreational behavior.

  USER BEHAVIOR
  ─────────────
  ► Subscribers make the majority of trips and dominate round-trip
    usage near transit stations.
  ► Customers (casual users) account for a higher proportion of
    round trips in recreational zones (e.g., Central Park, waterfront).
  ► Young riders (<25) take the longest trips on average;
    Seniors (≥60) take shorter, slower trips.
  ► Gender analysis (Welch t-test, α=0.05):
    - Trip duration difference is {gender_duration_sig}.
    - Trip speed difference is    {gender_speed_sig}.
    This is a formal statistical test (not just visual comparison),
    meaning the observed differences are unlikely due to chance.

  STATION & GEOGRAPHIC PATTERNS
  ──────────────────────────────
  ► Subscribers end trips in Financial/Midtown zones (business hubs,
    transit nodes) — consistent with commute-to-work usage.
  ► Customers end trips in Uptown/Recreational zones (parks, tourist
    attractions) — consistent with leisure riding.
  ► Top station pairs reveal high-demand commuter corridors that
    experience consistent daily ridership volume.

  BIKE UTILIZATION
  ────────────────
  ► A small subset of bike IDs accumulate disproportionately high
    total trip seconds (top 5% flagged for maintenance review).
  ► Over-utilized bikes concentrated in high-traffic station clusters.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
4. BUSINESS DECISIONS SUPPORTED BY THIS ANALYSIS
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

  STATION EXPANSION
  ► Expand docking capacity at high-demand start/end station pairs,
    particularly the top commuter corridors identified in Query H.
  ► Add new stations near Uptown/Recreational zones to serve the
    growing casual customer segment on weekends.

  MAINTENANCE PLANNING
  ► Schedule preventive maintenance for bikes in the top 5% utilization
    tier (identified in Query F) before failures occur.
  ► Rotate high-use bikes from busy stations to reduce wear concentration.

  STAFFING & REBALANCING
  ► Deploy extra rebalancing staff at 7–9 AM and 4–7 PM on weekdays
    near Midtown and Financial District stations.
  ► Shift weekend staffing toward park-adjacent stations and tourist areas.

  MARKETING & ENGAGEMENT
  ► Target Young and Adult demographic segments for subscription
    conversion campaigns (longer trips = higher engagement potential).
  ► Design tourist-friendly casual plans for weekend customers concentrated
    in recreational zones.
  ► Gender-differentiated campaigns can be justified given statistically
    significant behavioral differences in speed and duration.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
5. MACHINE LEARNING — MODEL COMPARISON & BEST MODEL
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Task: Predict rider gender (Male=0, Female=1) from trip features.

  Model                 Accuracy     F1 Score
  ──────────────────────────────────────────────
  {results_pd.iloc[0]['model']:<22}{results_pd.iloc[0]['accuracy']:.4f}       {results_pd.iloc[0]['f1_score']:.4f}  ← BEST
  {results_pd.iloc[1]['model']:<22}{results_pd.iloc[1]['accuracy']:.4f}       {results_pd.iloc[1]['f1_score']:.4f}
  {results_pd.iloc[2]['model']:<22}{results_pd.iloc[2]['accuracy']:.4f}       {results_pd.iloc[2]['f1_score']:.4f}

  BEST MODEL: {best_model_name}
  ─────────────────────────────────────
  Why it performed best:
{best_model_explanation}

  ACCURACY vs INTERPRETABILITY TRADEOFF:
  ► Logistic Regression: Most interpretable (coefficient weights per
    feature) but limited by its linearity assumption.
  ► Decision Tree: Fully interpretable via visual tree paths, captures
    non-linear splits, but prone to overfitting without pruning.
  ► Random Forest: Best generalization through ensemble averaging,
    but least interpretable as a black-box. Feature importances
    are available as a proxy for explainability.

  MOST INFLUENTIAL FEATURES (from Random Forest importances):
  ► trip_duration_seconds — riding duration varies significantly by gender
  ► rider_age             — age interacts strongly with gender in this dataset
  ► trip_speed_kmh        — speed patterns differ between male/female riders
  ► start_hour            — time-of-day riding behavior differs by gender
  ► user_type             — subscriber vs customer correlates with gender

  Feature pipeline: StringIndexer → OneHotEncoder → VectorAssembler
                    → StandardScaler → Classifier
  Train/Test split: 80% / 20% (seed=42 for reproducibility)

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
END OF PROJECT SUMMARY
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
""")


╔══════════════════════════════════════════════════════════════════════╗
║         FINAL PROJECT SUMMARY — NYC CITI BIKE BIG DATA ANALYSIS     ║
║         German International University | Big Data & NoSQL, 2026    ║
╚══════════════════════════════════════════════════════════════════════╝

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
1. WHY SPARK WAS THE RIGHT TOOL FOR THIS PROJECT
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
The Citi Bike dataset is a high-volume, multi-million-row dataset that
would be impractical to process with single-machine tools like Pandas.

Apache Spark was chosen because:
  ✔ Distributed in-memory processing: Spark partitions data across
    executor cores, enabling parallel computation of aggregations,
    joins, and window functions that would OOM a single machine.
  ✔ Lazy evaluation: Spark builds a DAG of transformations and only
    executes when an action (.show(), .count(), .write()) is called,
    optim